# Bokmål/nynorsk alignment av lånekassens dokumenter med NB sBERT 

In [1]:
import pandas as pd

df = pd.read_json("lånekassen_data.json")
df

,doc_hash,lang,url,domain,date,mimetype,fulltext
0,b5fc6f81d5709f0051da1964c3900e7ca8df65f7,nno,http://lanekassen.no/globalassets/brosjyrer-fe...,lanekassen.no,2022-12-19 01:53:28,pdf,[Er du flyktning? Du blir rekna som flyktning ...
1,1fbe096fdfdd9916be1313006ac5cf44ac650231,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 01:57:14,pdf,"[, , Du kan bruke dette skjemaet dersom du tar..."
2,582aaf9a510b3bde21b5b3e13ffc3453d6ca8174,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 01:57:21,pdf,"[, , Det er viktig at du les informasjonen på ..."
3,32e23cf05e4db850a9e2f622c2edd0482572677e,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 02:00:45,pdf,[Nynorsk Skjema for lærlinglønn Kor stort bort...
4,4ea46648c7ce59fe0c39b45779ed6402d8b41369,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 02:01:01,pdf,[Nynorsk Skjema I – artikkelnr. 9001550 – nyno...
...,...,...,...,...,...,...,...
561,e29a43da7165d92d937c56416d50f563201ab5fa,nob,http://lanekassen.no/siteassets/skjemaer-og-fi...,lanekassen.no,2022-12-19 02:50:18,pdf,"[, , 01.01.2020 Storebrand Bank ASA 70 % Bolig..."
562,c120ee7f582c758ae392b8f90795a7c664c39e90,nob,http://lanekassen.no/siteassets/skjemaer-og-fi...,lanekassen.no,2022-12-19 02:50:23,pdf,"[, , 06.11.2019 Storebrand Bank ASA 70 % Bolig..."
563,553101a9b0a9fac935cf31e5dd59555d23ecbefe,nob,https://statistikk.lanekassen.no/globalassets/...,lanekassen.no,2022-12-19 03:10:41,pdf,"[, , Flyktningstipendet Mottakere av flyktning..."
564,1f14c7bcc564613f79be7a8cae1a946e27cfd755,nob,https://statistikk.lanekassen.no/globalassets/...,lanekassen.no,2022-12-19 03:10:44,pdf,"[, , Tall og fakta om Lånekassens kunder og or..."


In [2]:
assert len(set(df.doc_hash)) == len(df)

In [3]:
nynorske = df[df.lang == "nno"].copy()
nynorske.index = range(len(nynorske))

bokmålske = df[df.lang == "nob"].copy()
bokmålske.index = range(len(bokmålske))

len(nynorske), len(bokmålske)

(256, 310)

Lim sammen avsnittene i hvert dokument

In [4]:
nynorske["texts_joined"] = nynorske.fulltext.apply(lambda x: "\n".join(x))
bokmålske["texts_joined"] = bokmålske.fulltext.apply(lambda x: "\n".join(x))

Last inn fasit

In [5]:
hash_to_i_nn = {e.doc_hash: e.Index for e in nynorske.itertuples()}
hash_to_i_bm = {e.doc_hash: e.Index for e in bokmålske.itertuples()}

fasit = pd.read_csv("lanekassen_fasit.csv")
fasit_set = {(hash_to_i_nn[nn_doc_hash], hash_to_i_bm[bm_doc_hash]) for nn_doc_hash, bm_doc_hash in zip(fasit.nynorsk_doc_hash, fasit.bokmål_doc_hash)}

def compare_matches_to_fasit(matches):
    matches = {(i, match["corpus_id"]) for i, match in matches}

    hits = fasit_set.intersection(matches)
    misses = fasit_set - matches
    
    return (hits, misses)

Last inn modellen

In [6]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('NbAiLab/nb-sbert-base', device="cuda")

# Tell hvor mange dokumenter og avsnitt som er for lange for modellen

In [7]:
max_len = model.max_seq_length
tokenizer = model.tokenizer

nynorsk_documents_token_sequence_lengths = [len(tokenizer.tokenize(text)) for text in nynorske.texts_joined]

under_max_len = 0
over_max_len = 0
zero_len = 0
for token_len in nynorsk_documents_token_sequence_lengths:
    if token_len:
        if token_len > max_len:
            over_max_len += 1
        else:
            under_max_len += 1
    else:
        zero_len += 1

assert zero_len == 0

print(f"""
Av de {len(nynorsk_documents_token_sequence_lengths)} nynorske dokumentene er:
    {under_max_len} under NB sBERT sin makslengde 
    {over_max_len}  over NB sBERT sin makslengde 
Altså er {round(under_max_len/len(nynorsk_documents_token_sequence_lengths)*100, 2)}% av dokumentene under makslengden 
""")

bokmål_documents_token_sequence_lengths = [len(tokenizer.tokenize(text)) for text in bokmålske.texts_joined]

under_max_len = 0
over_max_len = 0
zero_len = 0
for token_len in bokmål_documents_token_sequence_lengths:
    if token_len:
        if token_len > max_len:
            over_max_len += 1
        else:
            under_max_len += 1
    else:
        zero_len += 1

assert zero_len == 0

print(f"""
Av de {len(bokmål_documents_token_sequence_lengths)} dokumentene på bokmål er:
    {under_max_len} under NB sBERT sin makslengde 
    {over_max_len}  over NB sBERT sin makslengde 
Altså er {round(under_max_len/len(bokmål_documents_token_sequence_lengths)*100, 2)}% av dokumentene under makslengden 
""")


Av de 256 nynorske dokumentene er:
    2 under NB sBERT sin makslengde 
    254  over NB sBERT sin makslengde 
Altså er 0.78% av dokumentene under makslengden 


Av de 310 dokumentene på bokmål er:
    1 under NB sBERT sin makslengde 
    309  over NB sBERT sin makslengde 
Altså er 0.32% av dokumentene under makslengden 



In [8]:
max_len = model.max_seq_length
tokenizer = model.tokenizer

token_sequence_lengths = df.fulltext.apply(lambda x: [len(tokenizer.tokenize(e)) for e in x])

under_max_len = 0
over_max_len = 0
zero_len = 0
for e in token_sequence_lengths:
    for token_len in e:
        if token_len:
            if token_len > max_len:
                over_max_len += 1
            else:
                under_max_len += 1
        else:
            zero_len += 1

print(f"""
Det er totalt {sum((under_max_len, over_max_len))} ikke-tomme avsnitt/setninger (og {zero_len} er tomme)
Av de ikke-tomme avsnittene er:
    {under_max_len} under NB sBERT sin makslengde 
    {over_max_len}  over NB sBERT sin makslengde 
Altså er {round(under_max_len/sum((under_max_len, over_max_len))*100, 2)}% av de ikke-tomme avsnittene under makslengden 
""")



Det er totalt 12491 ikke-tomme avsnitt/setninger (og 4377 er tomme)
Av de ikke-tomme avsnittene er:
    8877 under NB sBERT sin makslengde 
    3614  over NB sBERT sin makslengde 
Altså er 71.07% av de ikke-tomme avsnittene under makslengden 



# Dokumentalignment 
Finn den likeste bokmål-dokument-embeddingen for hver nynorsk-dokument-embedding.  

In [9]:
from pathlib import Path 

def write_doc_matches_to_file(matches, filename):
    file_path = Path(filename)
    file_path.parent.mkdir(exist_ok=True, parents=True)
    
    nn_i, bm_i = zip(*[(i, res["corpus_id"]) for i, res in matches])
                    
    nn_hashes = list(nynorske.doc_hash.iloc[list(nn_i)])
    bm_hashes = list(bokmålske.doc_hash.iloc[list(bm_i)])

    pd.DataFrame({"nynorsk_doc_hash": nn_hashes, "bokmål_doc_hash": bm_hashes}).to_csv(filename, index=False)

In [10]:
doc_alignment_results = {}

## Aksepter cut-off
Send dokumentet as is til modellen (vil kuttes av på modellens makslengde)

In [11]:
import numpy as np
from pathlib import Path

base_path = "nb_sbert/texts_joined"

emb_path = Path(f"embeddings/{base_path}.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    bokmål_embeddings = model.encode(bokmålske.texts_joined)
    nynorsk_embeddings = model.encode(nynorske.texts_joined)
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]
write_doc_matches_to_file(matches, f"output/{base_path}.csv")

hits, misses = compare_matches_to_fasit(matches)
doc_alignment_results["naiv_cutoff"] = {"matches": len(matches), "threshold": threshold, "percent of docs": round(len(matches)/len(nynorske)*100, 2), "hits": len(hits), "misses": len(misses)}

### Inspiser resultater

In [12]:
from utils import print_matches, print_misses

# print_matches(matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_matches(non_matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_misses(misses, search_result, bokmålske.texts_joined, nynorske.texts_joined, bokmål_embeddings, nynorsk_embeddings, threshold)

#### Falsk positiv
Eksempel på en falsk positiv.  
Noen lister av datoer og banker og renter blir veldig like (similarity score er ~0.98)

In [13]:
i = 19 
print(search_result[i])
print(nynorske.texts_joined[i][:250])
print("\n___________\n")
print(bokmålske.texts_joined[search_result[i][0]["corpus_id"]][:250])

[{'corpus_id': 291, 'score': 0.9755688309669495}]


04.05.2022 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1,79% 04.05.2022 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1,97% 04.05.2022 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 2,07% 04.05.2022 Sunndal Sparebank B

___________



02.03.2022 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1.79% 02.03.2022 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 1.81% 02.03.2022 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1.97% 02.03.2022 KLP Banken AS Bolig


## Del opp dokumentene i biter mindre enn modellens makslengde og aggreger

In [14]:
from utils import tokenize_and_split_text

nynorsk_maxlen_parts = [tokenize_and_split_text(text, tokenizer=model.tokenizer, model_max_len=model.max_seq_length) for text in nynorske.texts_joined]
bokmål_maxlen_parts = [tokenize_and_split_text(text, tokenizer=model.tokenizer, model_max_len=model.max_seq_length) for text in bokmålske.texts_joined]

In [15]:
# from collections import Counter 
# pd.DataFrame(Counter([len(e) for e in nynorsk_maxlen_parts]).most_common(), columns=("antall lister", "antall dokumenter")).sort_values("antall lister")
# pd.DataFrame(Counter([len(e) for e in bokmål_maxlen_parts]).most_common(), columns=("antall lister", "antall dokumenter")).sort_values("antall lister")

### Mean pooling

In [16]:
import numpy as np 

base_path = "nb_sbert/maxlen_parts_mean_pooling"

emb_path = Path(f"embeddings/{base_path}.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = np.array([np.mean(model.encode(text_parts), axis=0) for text_parts in nynorsk_maxlen_parts])
    bokmål_embeddings = np.array([np.mean(model.encode(text_parts), axis=0) for text_parts in bokmål_maxlen_parts])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]
write_doc_matches_to_file(matches, f"output/{base_path}.csv")

hits, misses = compare_matches_to_fasit(matches)
doc_alignment_results["mean_pooling_biter"] = {"matches": len(matches), "threshold": threshold, "percent of docs": round(len(matches)/len(nynorske)*100, 2), "hits": len(hits), "misses": len(misses)}

### Inspiser resultater

In [17]:
# print_matches(matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_matches(non_matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_misses(misses, search_result, bokmålske.texts_joined, nynorske.texts_joined, bokmål_embeddings, nynorsk_embeddings, threshold)

#### Falsk positiv:

In [18]:
i = 19
print(search_result[i])
print(nynorske.texts_joined[i][:250])
print("\n___________\n")
print(bokmålske.texts_joined[search_result[i][0]["corpus_id"]][:250])

[{'corpus_id': 291, 'score': 0.9914844036102295}]


04.05.2022 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1,79% 04.05.2022 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1,97% 04.05.2022 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 2,07% 04.05.2022 Sunndal Sparebank B

___________



02.03.2022 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1.79% 02.03.2022 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 1.81% 02.03.2022 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1.97% 02.03.2022 KLP Banken AS Bolig


### Max pooling

In [19]:
import numpy as np 
base_path = "nb_sbert/maxlen_parts_max_pooling"

emb_path = Path(f"embeddings/{base_path}.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = np.array([np.max(model.encode(text_parts), axis=0) for text_parts in nynorsk_maxlen_parts])
    bokmål_embeddings = np.array([np.max(model.encode(text_parts), axis=0) for text_parts in bokmål_maxlen_parts])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]
write_doc_matches_to_file(matches, f"output/{base_path}.csv")

hits, misses = compare_matches_to_fasit(matches)
doc_alignment_results["max_pooling_biter"] = {"matches": len(matches), "threshold": threshold, "percent of docs": round(len(matches)/len(nynorske)*100, 2), "hits": len(hits), "misses": len(misses)}

### Inspiser resultater

In [20]:
# print_matches(matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_matches(non_matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_misses(misses, search_result, bokmålske.texts_joined, nynorske.texts_joined, bokmål_embeddings, nynorsk_embeddings, threshold)

#### Falsk positiv:
Den samme som over

In [21]:
i = 19
print(search_result[i])
print(nynorske.texts_joined[i][:250])
print("\n___________\n")
print(bokmålske.texts_joined[search_result[i][0]["corpus_id"]][:250])

[{'corpus_id': 293, 'score': 0.9619422554969788}]


04.05.2022 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1,79% 04.05.2022 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1,97% 04.05.2022 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 2,07% 04.05.2022 Sunndal Sparebank B

___________



03.11.2021 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1,29 % 03.11.2021 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1,47 % 03.11.2021 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 1,56 % 03.11.2021 Sunndal Spareban


# Setnings/avsnittsalignment

In [22]:
sent_alignment_results = {}

In [23]:
from collections import defaultdict

nynorske_sentences = defaultdict(list)
for t, df_ in nynorske.explode("fulltext").groupby("fulltext"):
    if len(t)>2:
        nynorske_sentences["text"].append(t)
        nynorske_sentences["doc_hashes"].append(set(df_.doc_hash))
        nynorske_sentences["urls"].append(set(df_.url))

nynorske_flat = pd.DataFrame(nynorske_sentences)

bokmålske_sentences = defaultdict(list)
for t, df_ in bokmålske.explode("fulltext").groupby("fulltext"):
    if len(t)>2:
        bokmålske_sentences["text"].append(t)
        bokmålske_sentences["doc_hashes"].append(set(df_.doc_hash))
        bokmålske_sentences["urls"].append(set(df_.url))

bokmålske_flat = pd.DataFrame(bokmålske_sentences)

In [24]:
len(bokmålske_flat), len(nynorske_flat)

(6345, 4026)

In [25]:
def write_sent_matches_to_file(matches, filename):
    nn_i, bm_i = zip(*[(i, res["corpus_id"]) for i, res in matches])

    nn = nynorske_flat.iloc[list(nn_i)].rename(mapper=lambda x: "nn_"+x, axis=1)
    bm = bokmålske_flat.iloc[list(bm_i)].rename(mapper=lambda x: "bm_"+x, axis=1)
    nn.index = range(len(nn))
    bm.index = range(len(bm))
    
    pd.concat([nn, bm], axis=1).to_csv(filename, index=False)


In [26]:
same_text = bokmålske_flat.merge(nynorske_flat, on="text", suffixes=["_bm", "_nn"])
sent_alignment_results["string_comparison"] = {"matches": len(same_text), "percent of sents": round(len(same_text)/len(nynorske_flat), 2)}

## Aksepter cut-off
Godta cut-off på sBERT sin maxlengde

In [27]:
import numpy as np

base_path = "nb_sbert/texts_flat"

emb_path = Path(f"embeddings/{base_path}.npz")

if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = model.encode(nynorske_flat.text)
    bokmål_embeddings = model.encode(bokmålske_flat.text)
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)

threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]

write_sent_matches_to_file(matches, f"output/{base_path}.csv")
sent_alignment_results["naiv_cutoff"] = {"threshold": threshold, "matches": len(matches), "percent of sents": round(len(matches)/len(nynorske_flat.text)*100, 2)}


### Inspiser resultater

In [28]:
# print_matches(matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=30)
# print_matches(non_matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=30)

## Del opp setninger/avsnitt som er lengre enn sBERT sin maxlengde og aggreger

In [29]:
from utils import tokenize_and_split_text

nynorsk_maxlen_parts_flat = [tokenize_and_split_text(text, model.tokenizer, model.max_seq_length) for text in nynorske_flat.text]
bokmål_maxlen_parts_flat = [tokenize_and_split_text(text, model.tokenizer, model.max_seq_length) for text in bokmålske_flat.text]

### Mean pooling

In [32]:
base_path = "nb_sbert/maxlen_parts_flat_mean_pooling"

emb_path = Path(f"embeddings/{base_path}.npz")

if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = np.array([np.mean(model.encode(sent), axis=0) for sent in nynorsk_maxlen_parts_flat])
    bokmål_embeddings = np.array([np.mean(model.encode(sent), axis=0) for sent in bokmål_maxlen_parts_flat])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)
    
search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)

threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]

write_sent_matches_to_file(matches, f"output/{base_path}.csv")
sent_alignment_results["mean_pooling_biter"] = {"threshold": threshold, "matches": len(matches), "percent of sents": round(len(matches)/len(nynorsk_maxlen_parts_flat)*100, 2)}

#### Inspiser resultater

In [33]:
# print_matches(matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=20)
# print_matches(non_matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=20)

### Max pooling

In [34]:
base_path = "nb_sbert/maxlen_parts_flat_max_pooling"

emb_path = Path(f"embeddings/{base_path}.npz")

if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = np.array([np.max(model.encode(sent), axis=0) for sent in nynorsk_maxlen_parts_flat])
    bokmål_embeddings = np.array([np.max(model.encode(sent), axis=0) for sent in bokmål_maxlen_parts_flat])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)

threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]

write_sent_matches_to_file(matches, f"output/{base_path}.csv")
sent_alignment_results["max_pooling_biter"] = {"threshold": threshold, "matches": len(matches), "percent of sents": round(len(matches)/len(nynorsk_maxlen_parts_flat)*100, 2)}

#### Inspiser resultater

In [35]:
# print_matches(matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=50)
# print_matches(non_matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=50)

#### Falsk negativ

In [36]:
i = 90
print(search_result[i])
print(nynorske_flat.text[i])
print("\n___________\n")
print(bokmålske_flat.text[search_result[i][0]["corpus_id"]])

[{'corpus_id': 69, 'score': 0.9206681251525879}]
18.03.2020 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 2,68 % 18.03.2020 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 2,73 % 18.03.2020 Sbanken ASA Boliglån 50 % 2,75 % 18.03.2020 Fana Sparebank Nettlån Bolig 50 % 2,80 % 18.03.2020 Nordea Direct Boliglån inntil 50 % 2,91 %

___________

03.08.2022 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 2,05% 03.08.2022 Sunndal Sparebank Boliglån 50 prosent 2,38% 03.08.2022 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 2,58% 03.08.2022 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 2,68% 03.08.2022 Nordea Direct Boliglån inntil 50 % 2,75%


#  Utvid makslengde
Selv om NB-sbert har en makslengde på 75 tokens, har den underliggende BERT-modellen en makslengde på 512.  
Vi eksperimenterer med å øke makslengden til 512 og se hva slags resulater vi får 
 

In [37]:
basemodel_max_len = model[0].auto_model.config.max_position_embeddings
model.max_seq_length = basemodel_max_len
model.get_max_seq_length()

512

## Tell hvor mange dokumenter og avsnitt som er for lange nå

In [38]:
max_len = model.max_seq_length
tokenizer = model.tokenizer

nynorsk_documents_token_sequence_lengths = [len(tokenizer.tokenize(text)) for text in nynorske.texts_joined]

under_max_len = 0
over_max_len = 0
zero_len = 0
for token_len in nynorsk_documents_token_sequence_lengths:
    if token_len:
        if token_len > max_len:
            over_max_len += 1
        else:
            under_max_len += 1
    else:
        zero_len += 1

assert zero_len == 0

print(f"""
Av de {len(nynorsk_documents_token_sequence_lengths)} nynorske dokumentene er:
    {under_max_len} under NB BERT sin makslengde 
    {over_max_len}  over NB BERT sin makslengde 
Altså er {round(under_max_len/len(nynorsk_documents_token_sequence_lengths)*100, 2)}% av dokumentene under makslengden 
""")

bokmål_documents_token_sequence_lengths = [len(tokenizer.tokenize(text)) for text in bokmålske.texts_joined]

under_max_len = 0
over_max_len = 0
zero_len = 0
for token_len in bokmål_documents_token_sequence_lengths:
    if token_len:
        if token_len > max_len:
            over_max_len += 1
        else:
            under_max_len += 1
    else:
        zero_len += 1

assert zero_len == 0

print(f"""
Av de {len(bokmål_documents_token_sequence_lengths)} dokumentene på bokmål er:
    {under_max_len} under NB BERT sin makslengde 
    {over_max_len}  over NB BERT sin makslengde 
Altså er {round(under_max_len/len(bokmål_documents_token_sequence_lengths)*100, 2)}% av dokumentene under makslengden 
""")


Av de 256 nynorske dokumentene er:
    114 under NB BERT sin makslengde 
    142  over NB BERT sin makslengde 
Altså er 44.53% av dokumentene under makslengden 


Av de 310 dokumentene på bokmål er:
    122 under NB BERT sin makslengde 
    188  over NB BERT sin makslengde 
Altså er 39.35% av dokumentene under makslengden 



In [39]:
max_len = model.max_seq_length
tokenizer = model.tokenizer

token_sequence_lengths = df.fulltext.apply(lambda x: [len(tokenizer.tokenize(e)) for e in x])

under_max_len = 0
over_max_len = 0
zero_len = 0
for e in token_sequence_lengths:
    for token_len in e:
        if token_len:
            if token_len > max_len:
                over_max_len += 1
            else:
                under_max_len += 1
        else:
            zero_len += 1

print(f"""
Det er totalt {sum((under_max_len, over_max_len))} ikke-tomme avsnitt/setninger (og {zero_len} er tomme)
Av de ikke-tomme avsnittene er:
    {under_max_len} under NB BERT sin makslengde 
    {over_max_len}  over NB BERT sin makslengde 
Altså er {round(under_max_len/sum((under_max_len, over_max_len))*100, 2)}% av de ikke-tomme avsnittene under makslengden 
""")



Det er totalt 12491 ikke-tomme avsnitt/setninger (og 4377 er tomme)
Av de ikke-tomme avsnittene er:
    12086 under NB BERT sin makslengde 
    405  over NB BERT sin makslengde 
Altså er 96.76% av de ikke-tomme avsnittene under makslengden 



## Dokumentalignment

In [40]:
import numpy as np
from pathlib import Path

base_path = "nb_sbert_extend/texts_joined"
emb_path = Path(f"embeddings/{base_path}.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    bokmål_embeddings = model.encode(bokmålske.texts_joined)
    nynorsk_embeddings = model.encode(nynorske.texts_joined)
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]
write_doc_matches_to_file(matches, f"output/{base_path}.csv")

hits, misses = compare_matches_to_fasit(matches)
doc_alignment_results["utvidet_naiv_cutoff"] = {"matches": len(matches), "threshold": threshold, "percent of docs": round(len(matches)/len(nynorske)*100, 2), "hits": len(hits), "misses": len(misses)}

In [41]:
from utils import print_matches, print_misses

# print_matches(matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_matches(non_matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_misses(misses, search_result, bokmålske.texts_joined, nynorske.texts_joined, bokmål_embeddings, nynorsk_embeddings, threshold)

### Del opp avsnittene i biter mindre enn modellens makslengde og aggreger

In [42]:
from utils import tokenize_and_split_text

nynorsk_maxlen_parts = [tokenize_and_split_text(text, tokenizer=model.tokenizer, model_max_len=model.max_seq_length) for text in nynorske.texts_joined]
bokmål_maxlen_parts = [tokenize_and_split_text(text, tokenizer=model.tokenizer, model_max_len=model.max_seq_length) for text in bokmålske.texts_joined]

In [43]:
# from collections import Counter 
# pd.DataFrame(Counter([len(e) for e in nynorsk_maxlen_parts]).most_common(), columns=("antall lister", "antall dokumenter")).sort_values("antall lister")
# pd.DataFrame(Counter([len(e) for e in bokmål_maxlen_parts]).most_common(), columns=("antall lister", "antall dokumenter")).sort_values("antall lister")

### Aggreger setningsvektorene: Mean pooling

In [44]:
import numpy as np 

base_path = "nb_sbert_extend/maxlen_parts_mean_pooling"

emb_path = Path(f"embeddings/{base_path}.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = np.array([np.mean(model.encode(text_parts), axis=0) for text_parts in nynorsk_maxlen_parts])
    bokmål_embeddings = np.array([np.mean(model.encode(text_parts), axis=0) for text_parts in bokmål_maxlen_parts])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]
write_doc_matches_to_file(matches, f"output/{base_path}.csv")

hits, misses = compare_matches_to_fasit(matches)
doc_alignment_results["utvidet_mean_pooling_biter"] = {"matches": len(matches), "threshold": threshold, "percent of docs": round(len(matches)/len(nynorske)*100, 2), "hits": len(hits), "misses": len(misses)}

In [45]:
# print_matches(matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_matches(non_matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_misses(misses, search_result, bokmålske.texts_joined, nynorske.texts_joined, bokmål_embeddings, nynorsk_embeddings, threshold)

### Aggreger setningsvektorene: Max pooling

In [46]:
import numpy as np 

base_path = "nb_sbert_extend/maxlen_parts_max_pooling"

emb_path = Path(f"embeddings/{base_path}.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = np.array([np.max(model.encode(text_parts), axis=0) for text_parts in nynorsk_maxlen_parts])
    bokmål_embeddings = np.array([np.max(model.encode(text_parts), axis=0) for text_parts in bokmål_maxlen_parts])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]
write_doc_matches_to_file(matches, f"output/{base_path}.csv")

hits, misses = compare_matches_to_fasit(matches)
doc_alignment_results["utvidet_max_pooling_biter"] = {"matches": len(matches), "threshold": threshold, "percent of docs": round(len(matches)/len(nynorske)*100, 2), "hits": len(hits), "misses": len(misses)}

In [47]:
# print_matches(matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_matches(non_matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_misses(misses, search_result, bokmålske.texts_joined, nynorske.texts_joined, bokmål_embeddings, nynorsk_embeddings, threshold)

## Setningsalignment

### Naiv approach
Godta cut-off på NB BERT sin maxlengde

In [48]:
import numpy as np

base_path = "nb_sbert_extend/texts_flat"

emb_path = Path(f"embeddings/{base_path}.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = model.encode(nynorske_flat.text)
    bokmål_embeddings = model.encode(bokmålske_flat.text)
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)

threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]

write_sent_matches_to_file(matches, f"output/{base_path}.csv")
sent_alignment_results["utvidet_naiv_cutoff"] = {"threshold": threshold, "matches": len(matches), "percent of sents": round(len(matches)/len(nynorske_flat.text)*100, 2)}

In [49]:
# print_matches(matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=30)
# print_matches(non_matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=30)

### Del opp setninger/avsnitt som er lengre enn sBERT sin maxlengde

In [50]:
from utils import tokenize_and_split_text

nynorsk_maxlen_parts_flat = [tokenize_and_split_text(text, model.tokenizer, model.max_seq_length) for text in nynorske_flat.text]
bokmål_maxlen_parts_flat = [tokenize_and_split_text(text, model.tokenizer, model.max_seq_length) for text in bokmålske_flat.text]

### Mean pooling

In [51]:
base_path = "nb_sbert_extend/maxlen_parts_flat_mean_pooling"

emb_path = Path(f"embeddings/{base_path}.npz")

if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = np.array([np.mean(model.encode(sent), axis=0) for sent in nynorsk_maxlen_parts_flat])
    bokmål_embeddings = np.array([np.mean(model.encode(sent), axis=0) for sent in bokmål_maxlen_parts_flat])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)
    
search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)

threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]

write_sent_matches_to_file(matches, f"output/{base_path}.csv")
sent_alignment_results["utvidet_mean_pooling_biter"] = {"threshold": threshold, "matches": len(matches), "percent of sents": round(len(matches)/len(nynorsk_maxlen_parts_flat)*100, 2)}

In [52]:
# print_matches(matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=20)
# print_matches(non_matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=20)

### Max pooling

In [53]:

base_path = "nb_sbert_extend/maxlen_parts_flat_max_pooling"

emb_path = Path(f"embeddings/{base_path}.npz")

if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = np.array([np.max(model.encode(sent), axis=0) for sent in nynorsk_maxlen_parts_flat])
    bokmål_embeddings = np.array([np.max(model.encode(sent), axis=0) for sent in bokmål_maxlen_parts_flat])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)

threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]

write_sent_matches_to_file(matches, f"output/{base_path}.csv")
sent_alignment_results["utvidet_max_pooling_biter"] = {"threshold": threshold, "matches": len(matches), "percent of sents": round(len(matches)/len(nynorsk_maxlen_parts_flat)*100, 2)}

In [54]:
# print_matches(matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=50)
# print_matches(non_matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=50)

# Konklusjon


## Dokumentalignment

Metodene som treffer flest av dokumentene i fasiten er mean pooling over bitene og utvided versjon med naiv cutoff, som begge treffer 52/55 av dokumentene i fasiten.  
Mean pooling har 236 matches, mens naiv cutoff med utvidet versjon har 236.  
Siden vi ikke har noen god måte å måle falske positiver (annet enn stikkprøvder), er det ikke så godt å si hvilke av disse metodene som egentlig er best.  

In [55]:
pd.DataFrame(doc_alignment_results).T.sort_values("hits")

,matches,threshold,percent of docs,hits,misses
naiv_cutoff,204.0,0.95,79.69,42.0,13.0
utvidet_max_pooling_biter,215.0,0.95,83.98,46.0,9.0
max_pooling_biter,220.0,0.95,85.94,48.0,7.0
utvidet_mean_pooling_biter,230.0,0.95,89.84,50.0,5.0
utvidet_naiv_cutoff,232.0,0.95,90.62,52.0,3.0
mean_pooling_biter,236.0,0.95,92.19,52.0,3.0


## Setningsalignment
For setningsalignment fikk sBERT med utvidet makslengde flere matches enn den originale modellen på alle eksperimenter.  
Naiv cutoff gir flere matches enn aggregering, og vi vet at de aller fleste dokumentene er kortere enn makslengden.

In [56]:
pd.DataFrame(sent_alignment_results).T.sort_values("matches")

,matches,percent of sents,threshold
string_comparison,338.0,0.08,NaN
max_pooling_biter,2593.0,64.41,0.95
mean_pooling_biter,2633.0,65.40,0.95
naiv_cutoff,2687.0,66.74,0.95
utvidet_max_pooling_biter,2697.0,66.99,0.95
utvidet_mean_pooling_biter,2700.0,67.06,0.95
utvidet_naiv_cutoff,2701.0,67.09,0.95
